# Planning Pattern Extension - Planner Plus ReAct Executor


<img src="https://www.dailydoseofds.com/content/images/2026/01/https-3a-2f-2fsubstack-post-media-s3-amazonaws-com-2fpublic-2fimages-2f643b6891-84f6-4672-aa1f-4724c5ad2d12_716x526-3.gif" alt="Planning pattern" width="500"/>

This extension notebook is closer to Figure 1 from the Lab 4 instructions. Instead of manually inserting a later clue, we let one agent decide what evidence to check next, and let another agent carry out that one step with real notebook tools.

Keep these two roles separate as you work:
- `PlanningAgent` is the planner. It is like a coordinator that decides what should happen next.
- `ReactAgent` is the executor. Here, executor simply means the helper agent that carries out one evidence-gathering task.

When `ReactAgent` finishes using a tool, the tool returns a result. In this notebook, that returned result is called an `observation`. `PlanningAgent` then reads that observation and decides whether the plan should change.

This notebook is an advanced follow-up to `03_lab_notebook.ipynb`, not a replacement for it. It now has two parts: Part 1 is a guided two-round example, and Part 2 is an automatic multi-round demo.


## What You Will Do

This notebook has two parts:
- **Part 1** is a guided two-round example that shows the workflow slowly.
- **Part 2** is an automatic multi-round demo that can continue for up to `5` rounds or stop early when `PlanningAgent` returns `FINISHED:`.

In Part 1, you will:
1. Show all tools from the start.
2. Let `PlanningAgent` build the first plan.
3. Let `PlanningAgent` assign the first task to `ReactAgent`.
4. Let `ReactAgent` gather evidence for that task.
5. Feed that observation back to `PlanningAgent`.
6. Let `PlanningAgent` revise the plan and assign the next task.
7. Let `ReactAgent` gather the next piece of evidence.
8. Let `PlanningAgent` write the final report.

After Part 1, Part 2 reruns the same pattern automatically with a bounded loop.


## Lab Question and Response Format

Use the staged artifacts to answer this advanced planning question:

**What timeline best explains the missing-phone interval, what remaining dependency or uncertainty does the first ReactAgent result identify, and how should the explanation change after the next evidence check?**

In this notebook, the planner-control helper returns one of two short formats:
- `NEXT_TASK: ...` means `PlanningAgent` wants `ReactAgent` to gather one more piece of evidence.
- `FINISHED: ...` means `PlanningAgent` thinks the evidence is sufficient and the workflow can stop.

Part 2 uses this same `NEXT_TASK:` / `FINISHED:` control pattern automatically. In a fuller practical workflow, Steps 4-7 would repeat until `PlanningAgent` returns `FINISHED:` or the evidence is sufficient for a careful report.

All six tools are visible from the beginning. Even so, good planning should usually start with the most direct timeline records first, then move to broader checks only when the first observation reveals a remaining uncertainty.

By the end of the notebook, the final report should be organized with these sections:
1. Planner step log
2. Direct-evidence findings
3. Network evidence findings
4. Final timeline and evidence-bounded conclusion

Note for students: `search_network_evidence(...)` only searches the local `data/` folder in this lab. It does not search the web.


## Part 1: Guided Two-Round Example

Start with this slower walkthrough. It shows the planning workflow one step at a time before the notebook moves to the automatic demo in Part 2.


### Step 1: Set Up the Notebook

Run this setup cell first. It loads the Lab 4 configuration from `.env`, checks that you opened the notebook from the correct folder, and prepares the client, imports, and data path.

To run a cell in Jupyter, click the cell and press `Shift+Enter`.


In [ ]:
import csv
import json
import os
import string
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

# This setup cell assumes you opened the notebook from this lab folder.
# It loads this lab's .env, adds src/ to the import path, and prepares the case data.
LAB_NAME = 'lab4_planning_pattern'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(
        f'Open this notebook from the {LAB_NAME} folder.'
    )

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError(
        f'Expected .env in this folder. Copy .env.example to .env first.'
    )

src_dir = repo_root / 'src'
if str(src_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(src_dir.resolve()))

load_dotenv(env_path, override=True)

MODEL = os.getenv('MODEL')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f'MODEL or OLLAMA_BASE_URL is missing from {env_path}')

data_dir = lab_dir / 'data'
if not data_dir.exists():
    raise FileNotFoundError('Could not find the Lab 4 data folder')

from agentic_patterns.planning_pattern.planning_agent import PlanningAgent
from agentic_patterns.react_pattern.react_agent import ReactAgent
from agentic_patterns.tool_pattern.tool import tool

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)
print('Data files:', sorted(path.name for path in data_dir.iterdir() if path.is_file()))


### Step 2: Define the `ReactAgent` Tools

These tools are the evidence menu for `ReactAgent`.

In plain language: `PlanningAgent` makes the plan, but `ReactAgent` is the helper agent that actually uses the tools to open records, read timestamps, and bring back results.

In this simpler version of the notebook, all six tools are visible from the start:
- `list_artifact_files()`
- `get_incident_window()`
- `get_unlock_events()`
- `get_call_log()`
- `get_whatsapp_events()`
- `search_network_evidence(query)`

This does not mean `PlanningAgent` should jump straight to the search tool. The teaching goal is to start with the most direct timeline evidence first, then use broader search only if the first `ReactAgent` result shows a remaining uncertainty, such as connectivity.

Each tool returns a short structured summary so the notebook stays readable.


In [ ]:
# Helper function: read one CSV evidence file and return its rows.
def read_csv(filename: str) -> list[dict]:
    with (data_dir / filename).open(encoding='utf-8') as handle:
        return list(csv.DictReader(handle))


# Helper function: return one JSON string with indentation for readability.
def as_pretty_json(payload: dict | list) -> str:
    return json.dumps(payload, indent=2)


# Tool 1: show which evidence files are available in this staged case package.
@tool
def list_artifact_files() -> str:
    """Return the artifact files available in the Lab 4 data folder."""
    artifact_files = sorted(path.name for path in data_dir.iterdir() if path.is_file())
    return as_pretty_json({'artifact_files': artifact_files})


# Tool 2: return the main incident window from the case manifest.
@tool
def get_incident_window() -> str:
    """Return the UTC start and end timestamps for the missing-phone interval."""
    manifest = json.loads((data_dir / 'artifact_manifest.json').read_text(encoding='utf-8'))
    incident_window = manifest['incident_window_utc']
    return as_pretty_json(
        {
            'start': incident_window['start'],
            'end': incident_window['end'],
            'analysis_timezone': manifest['analysis_timezone'],
        }
    )


# Tool 3: return the unlock and lock events that bound device access in the case.
@tool
def get_unlock_events() -> str:
    """Return the recorded unlock and lock events for the device."""
    rows = read_csv('unlock_events.csv')
    return as_pretty_json({'event_count': len(rows), 'rows': rows})


# Tool 4: return the phone call log summary.
@tool
def get_call_log() -> str:
    """Return the recorded phone call events for the incident period."""
    rows = read_csv('call_log.csv')
    return as_pretty_json({'event_count': len(rows), 'rows': rows})


# Tool 5: return the WhatsApp activity summary.
@tool
def get_whatsapp_events() -> str:
    """Return the WhatsApp activity relevant to the incident period."""
    rows = read_csv('whatsapp_events.csv')
    return as_pretty_json({'event_count': len(rows), 'rows': rows})


# Tool 6: search the local evidence files for network-related clues.
@tool
def search_network_evidence(query: str) -> str:
    """Search the local Lab 4 data folder for network-related evidence that matches the query."""
    query_terms = {
        token.strip(string.punctuation).lower()
        for token in query.split()
        if token.strip(string.punctuation)
    }
    if not query_terms:
        query_terms = {'network'}

    matches = []
    for path in sorted(data_dir.iterdir()):
        if not path.is_file():
            continue
        file_matches = []
        for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), start=1):
            lower_line = line.lower()
            if any(term in lower_line for term in query_terms):
                file_matches.append({'line_number': line_number, 'text': line})
            if len(file_matches) >= 4:
                break
        if file_matches:
            matches.append({'file': path.name, 'matches': file_matches})

    if not matches:
        fallback_lines = (data_dir / 'network_status.csv').read_text(encoding='utf-8').splitlines()[:4]
        return as_pretty_json(
            {
                'query': query,
                'matches': [],
                'note': 'No exact keyword match found. Showing the start of network_status.csv as a fallback.',
                'fallback_excerpt': fallback_lines,
            }
        )

    return as_pretty_json({'query': query, 'matches': matches})


# All six tools are visible from the start in this simpler teaching version.
react_tools = [
    list_artifact_files,
    get_incident_window,
    get_unlock_events,
    get_call_log,
    get_whatsapp_events,
    search_network_evidence,
]

tool_schema_preview = {
    tool.name: json.loads(tool.fn_signature)
    for tool in react_tools
}

tool_schema_preview


### Step 3: Let `PlanningAgent` Build the Initial Investigation Plan

`PlanningAgent` starts from the case question plus the full artifact list. That means it can see from the beginning that call, WhatsApp, and network evidence all exist in this case.

Even so, good planning should usually begin with the most direct timeline evidence first. The point is not to hide tools. The point is to make the first plan prioritize the clearest records before moving to broader search.

As you read the plan, ask yourself: what should `PlanningAgent` check first, and what uncertainty might still remain after the first `ReactAgent` result?


In [ ]:
planning_agent = PlanningAgent(client=client, model=MODEL)

case_question = (
    'What timeline best explains the missing-phone interval, what remaining dependency or uncertainty does the first ReactAgent result identify, '
    'and how should the explanation change after the next evidence check?'
)

artifact_menu = {
    'artifact_files': sorted(path.name for path in data_dir.iterdir() if path.is_file()),
    'note': 'PlanningAgent should choose what ReactAgent needs to inspect first from the full visible tool set.',
}

planner_case_context = (
    f'{case_question}\n\n'
    f'High-level artifact list:\n{json.dumps(artifact_menu, indent=2)}'
)

initial_plan = planning_agent.build_initial_plan(planner_case_context)
display(Markdown('### PlanningAgent Initial Output\n\n' + initial_plan))


### Step 4: Ask `PlanningAgent` for the Next `ReactAgent` Task

This helper asks `PlanningAgent` for exactly one next step.

At this point, `PlanningAgent` has the case question, the full artifact list, and the names of all six tools that `ReactAgent` can use. `PlanningAgent` does not run those tools itself. Instead, it decides which one-step task `ReactAgent` should do next.

Even though all six tools are already visible, the first task should usually start with the most direct timeline-building evidence before broader search is used.


In [ ]:
PLANNER_CONTROLLER_SYSTEM_PROMPT = """
You coordinate a planning workflow for a digital forensics case.

Your job is to choose the next single evidence-gathering task for ReactAgent.

Return exactly one of these two formats:
NEXT_TASK: <one concrete task sentence>
FINISHED: <brief reason and final reporting instruction>

Rules:
- Ask for only one task at a time.
- The task must be solvable with the listed ReactAgent tools.
- Start with the most direct timeline-building evidence before broader search.
- Use network-related search only after direct evidence reveals a remaining dependency or uncertainty about connectivity, delivery timing, or online/offline status.
- Keep the task short and specific.
""".strip()


# Ask PlanningAgent for the next task using the current plan and current observations.
def ask_planner_for_next_task(
    case_question: str,
    current_plan: str,
    observations: str,
    tool_names: list[str],
) -> str:
    observation_text = observations if observations else 'No ReactAgent observations yet.'
    prompt = (
        f'Case question:\n{case_question}\n\n'
        f'Current plan:\n{current_plan}\n\n'
        f'Observations so far:\n{observation_text}\n\n'
        f'Available ReactAgent tools:\n- ' + '\n- '.join(tool_names)
    )
    return client.chat.completions.create(
        messages=[
            {'role': 'system', 'content': PLANNER_CONTROLLER_SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ],
        model=MODEL,
    ).choices[0].message.content.strip()


# Pull the task text out of a PlanningAgent response such as "NEXT_TASK: ...".
def extract_next_task(planner_output: str) -> str | None:
    if 'NEXT_TASK:' in planner_output:
        return planner_output.split('NEXT_TASK:', 1)[1].strip()
    if planner_output.startswith('FINISHED:'):
        return None
    return planner_output.strip()


planner_decision_1 = ask_planner_for_next_task(
    case_question=case_question,
    current_plan=initial_plan,
    observations='',
    tool_names=[tool.name for tool in react_tools],
)
display(Markdown('### PlanningAgent Decision for ReactAgent Round 1\n\n' + planner_decision_1))

next_task_1 = extract_next_task(planner_decision_1)
print('Round 1 task:', next_task_1)


### Step 5: Run `ReactAgent` on the First Assigned Task

Now `ReactAgent` carries out `PlanningAgent`'s first task using the same full tool set from Step 2.

Even though all tools are available from the start, a good first task should usually focus on the clearest timeline evidence first. After that first result comes back, `PlanningAgent` can decide whether another evidence check is needed.

After you run the cell, look for two things:
- what evidence was gathered
- what remaining uncertainty the result identifies


In [ ]:
round_1_react_agent = ReactAgent(tools=react_tools, client=client, model=MODEL)

react_agent_prompt_1 = f"""
Carry out this PlanningAgent-assigned task using the available tools:

{next_task_1}

Requirements:
- Focus first on the most direct evidence needed for this assigned task.
- Mention the key timestamps you find.
- If the case still has a remaining dependency or uncertainty, say what it is.
- Return a short response with these labels:
  Direct evidence gathered:
  Remaining dependency or uncertainty:
""".strip()

react_agent_result_1 = round_1_react_agent.run(user_msg=react_agent_prompt_1)
display(Markdown('### ReactAgent Round 1 Final Response\n\n' + react_agent_result_1))


### Step 6: Feed the First `ReactAgent` Result Back to `PlanningAgent`

Now `PlanningAgent` reads a real result from `ReactAgent` instead of a manually inserted note.

This is an important planning-pattern moment: `PlanningAgent` is not guessing in the dark anymore. It is revising the plan based on evidence that was actually gathered.

Round 2 happens because of this first observation. If the first result still leaves a remaining uncertainty, `PlanningAgent` can now choose the next evidence check from the same full tool set.


In [ ]:
revised_plan_1 = planning_agent.revise_plan(
    user_msg=planner_case_context,
    current_plan=initial_plan,
    observations=f'ReactAgent round 1 result:\n{react_agent_result_1}',
)
display(Markdown('### PlanningAgent Revised Output After Round 1\n\n' + revised_plan_1))

planner_decision_2 = ask_planner_for_next_task(
    case_question=case_question,
    current_plan=revised_plan_1,
    observations=react_agent_result_1,
    tool_names=[tool.name for tool in react_tools],
)
display(Markdown('### PlanningAgent Decision for ReactAgent Round 2\n\n' + planner_decision_2))

next_task_2 = extract_next_task(planner_decision_2)
print('Round 2 task:', next_task_2)


### Step 7: Run `ReactAgent` on the Next Evidence Check

This second `ReactAgent` round uses the same full tool set as round 1.

The difference is not new tool availability. The difference is that `PlanningAgent` has now read the first observation and can choose a better next check. If the remaining uncertainty is about connectivity, this is the point where `search_network_evidence(...)` may become the right tool to use.

After you run the cell, check whether the result clearly explains what changed after this second evidence check.


In [ ]:
round_2_react_agent = ReactAgent(tools=react_tools, client=client, model=MODEL)

react_agent_prompt_2 = f"""
Carry out this PlanningAgent-assigned task using the available tools:

{next_task_2}

Requirements:
- Use whichever available tool best matches the assigned task.
- If the remaining uncertainty is about connectivity, `search_network_evidence(...)` is a reasonable tool to use.
- Report the important file name, timestamps, and status details you find.
- Return a short response with these labels:
  Network evidence gathered:
  Impact on the timeline:
""".strip()

react_agent_result_2 = round_2_react_agent.run(user_msg=react_agent_prompt_2)
display(Markdown('### ReactAgent Round 2 Final Response\n\n' + react_agent_result_2))


### Step 8: Let `PlanningAgent` Write the Final Report

`PlanningAgent` now has three things:
- the original plan
- the first `ReactAgent` result
- the second `ReactAgent` result from the next evidence check

In this teaching notebook, we stop after two `ReactAgent` results and write the final report. In a more practical workflow, `PlanningAgent` might continue the loop, request another observation, and wait to return `FINISHED:` before writing the report.

A strong final report should connect the steps together clearly: what was checked first, what remaining uncertainty was identified, what the next evidence check added, and why the final conclusion must stay evidence-bounded.


In [ ]:
combined_observations = (
    'ReactAgent round 1 result:\n'
    f'{react_agent_result_1}\n\n'
    'ReactAgent round 2 result:\n'
    f'{react_agent_result_2}'
)

revised_plan_2 = planning_agent.revise_plan(
    user_msg=planner_case_context,
    current_plan=revised_plan_1,
    observations=combined_observations,
)
display(Markdown('### PlanningAgent Update After Round 2\n\n' + revised_plan_2))

final_report_prompt = (
    'Use the case question, the current investigation plan, and the two ReactAgent observations below to write the final forensic report.\n\n'
    'Return a report with:\n'
    '1. Planner step log\n'
    '2. Direct-evidence findings\n'
    '3. Network evidence findings\n'
    '4. Final timeline and evidence-bounded conclusion\n\n'
    'The second observation is the next evidence check chosen after the first observation.\n\n'
    f'Case question:\n{case_question}\n\n'
    f'Current plan:\n{revised_plan_2}\n\n'
    f'ReactAgent observations:\n{combined_observations}'
)

final_report = planning_agent._complete(final_report_prompt)
display(Markdown('### PlanningAgent Final Report\n\n' + final_report))


## Part 2: Automatic Multi-Round Demo

Part 1 walked through the workflow slowly. Part 2 reruns the same planning pattern with fresh state and lets the notebook continue automatically for up to `5` rounds.

This is still a bounded classroom demo. The loop stops early if `PlanningAgent` returns `FINISHED:`, if no usable next task is returned, or if the planner repeats the same task twice in a row.


### Step 9: Define the Automatic Loop Helper

This helper rebuilds the case from scratch so Part 2 does not depend on Part 1 variables. It collects round-by-round summaries, captures the raw `ReactAgent` trace quietly, and returns one result object for display and later instructor inspection.


In [ ]:
from contextlib import redirect_stdout
from io import StringIO

AUTO_MAX_ROUNDS = 5

AUTO_CASE_QUESTION = (
    'What timeline best explains the missing-phone interval, what remaining dependency or uncertainty does the first ReactAgent result identify, '
    'and how should the explanation change after the next evidence check?'
)


# Helper function: rebuild the case question and artifact list for the automatic demo.
def build_automatic_case_context() -> tuple[str, str]:
    artifact_menu = {
        'artifact_files': sorted(path.name for path in data_dir.iterdir() if path.is_file()),
        'note': 'PlanningAgent should choose what ReactAgent needs to inspect next from the full visible tool set.',
    }
    planner_case_context = (
        f'{AUTO_CASE_QUESTION}\n\n'
        f'High-level artifact list:\n{json.dumps(artifact_menu, indent=2)}'
    )
    return AUTO_CASE_QUESTION, planner_case_context


# Helper function: keep the automatic-round instructions consistent from one round to the next.
def build_automatic_round_prompt(task: str) -> str:
    return f"""
Carry out this PlanningAgent-assigned task using the available tools:

{task}

Requirements:
- Carry out only this assigned task.
- Use the available tool or tools that best match the task.
- Mention the important file names, timestamps, and status details you find.
- Return a short response with these labels:
  Evidence gathered:
  Impact on the plan:
""".strip()


# Helper function: normalize task text so repeated-task detection is less brittle.
def normalize_task(task: str) -> str:
    return ' '.join(task.split())


# Helper function: turn the collected round results into one observation block for PlanningAgent.
def format_cumulative_observations(round_logs: list[dict]) -> str:
    if not round_logs:
        return 'No ReactAgent observations were collected.'
    return '\n\n'.join(
        [
            f"ReactAgent round {log['round_number']} result:\n{log['react_response']}"
            for log in round_logs
        ]
    )


# Helper function: build the final-report prompt from the latest plan and all collected observations.
def build_automatic_final_report_prompt(
    case_question: str,
    current_plan: str,
    round_logs: list[dict],
    stop_reason: str,
) -> str:
    combined_observations = format_cumulative_observations(round_logs)
    return (
        'Use the case question, the current investigation plan, and the ReactAgent observations below to write the final forensic report.\n\n'
        'Return a report with:\n'
        '1. Planner step log\n'
        '2. Direct-evidence findings\n'
        '3. Network evidence findings\n'
        '4. Final timeline and evidence-bounded conclusion\n\n'
        f'Loop stop reason:\n{stop_reason}\n\n'
        f'Case question:\n{case_question}\n\n'
        f'Current plan:\n{current_plan}\n\n'
        f'ReactAgent observations:\n{combined_observations}'
    )


# Main helper: run the planning loop automatically for a bounded number of rounds.
def run_automatic_planner_react_demo(max_rounds: int = AUTO_MAX_ROUNDS) -> dict:
    case_question, planner_case_context = build_automatic_case_context()
    planning_agent = PlanningAgent(client=client, model=MODEL)
    initial_plan = planning_agent.build_initial_plan(planner_case_context)
    current_plan = initial_plan

    round_logs = []
    previous_task = None
    last_planner_decision = ''
    stop_reason = ''

    for round_number in range(1, max_rounds + 1):
        last_planner_decision = ask_planner_for_next_task(
            case_question=case_question,
            current_plan=current_plan,
            observations=format_cumulative_observations(round_logs),
            tool_names=[tool.name for tool in react_tools],
        )

        if last_planner_decision.startswith('FINISHED:'):
            stop_reason = (
                f'PlanningAgent returned FINISHED after {round_number - 1} completed round(s).'
            )
            break

        next_task = extract_next_task(last_planner_decision)
        if not next_task:
            stop_reason = (
                f'PlanningAgent did not return a usable NEXT_TASK after {round_number - 1} completed round(s).'
            )
            break

        normalized_task = normalize_task(next_task)
        if previous_task is not None and normalized_task == previous_task:
            stop_reason = (
                f'PlanningAgent repeated the same task in round {round_number}, so the demo stopped early to avoid a stuck loop.'
            )
            break

        # Create a fresh ReactAgent each round so the system prompt does not accumulate repeatedly.
        round_react_agent = ReactAgent(tools=react_tools, client=client, model=MODEL)
        trace_buffer = StringIO()
        with redirect_stdout(trace_buffer):
            react_response = round_react_agent.run(
                user_msg=build_automatic_round_prompt(next_task)
            )

        combined_observations = format_cumulative_observations(
            round_logs
            + [
                {
                    'round_number': round_number,
                    'react_response': react_response,
                }
            ]
        )
        revised_plan = planning_agent.revise_plan(
            user_msg=planner_case_context,
            current_plan=current_plan,
            observations=combined_observations,
        )

        round_logs.append(
            {
                'round_number': round_number,
                'planner_decision': last_planner_decision,
                'task': next_task,
                'react_response': react_response,
                'revised_plan': revised_plan,
                'react_trace': trace_buffer.getvalue(),
            }
        )
        current_plan = revised_plan
        previous_task = normalized_task
    else:
        stop_reason = (
            f'Reached the automatic cap of {max_rounds} rounds before PlanningAgent returned FINISHED.'
        )

    final_report = planning_agent._complete(
        build_automatic_final_report_prompt(
            case_question=case_question,
            current_plan=current_plan,
            round_logs=round_logs,
            stop_reason=stop_reason,
        )
    )

    return {
        'case_question': case_question,
        'planner_case_context': planner_case_context,
        'initial_plan': initial_plan,
        'round_logs': round_logs,
        'stop_reason': stop_reason,
        'final_plan': current_plan,
        'final_planner_decision': last_planner_decision,
        'final_report': final_report,
    }


# Helper function: format the automatic demo results as readable notebook Markdown.
def format_automatic_demo_result(result: dict) -> str:
    sections = [
        '### Automatic Demo Initial Plan\n\n' + result['initial_plan'],
    ]

    for log in result['round_logs']:
        sections.append(
            f"### Automatic Round {log['round_number']}\n\n"
            f"**PlanningAgent decision**\n\n{log['planner_decision']}\n\n"
            f"**Assigned task**\n\n`{log['task']}`\n\n"
            f"**ReactAgent final response**\n\n{log['react_response']}\n\n"
            f"**Revised plan after round {log['round_number']}**\n\n{log['revised_plan']}"
        )

    sections.append('### Automatic Loop Stop Reason\n\n' + result['stop_reason'])
    sections.append('### Automatic Demo Final Report\n\n' + result['final_report'])
    sections.append(
        "Raw React traces are stored in `auto_demo_result['round_logs'][i]['react_trace']` if you want to inspect them later."
    )
    return '\n\n'.join(sections)


### Step 10: Run the Automatic Multi-Round Demo

This cell runs the bounded loop with fresh state. It can stop early when `PlanningAgent` returns `FINISHED:`, when no usable next task is returned, or when the planner repeats the same task twice in a row.

The main output shows only readable round summaries and the final report. Raw `ReactAgent` traces are kept inside `auto_demo_result` for later inspection if you want them.


In [ ]:
auto_demo_result = run_automatic_planner_react_demo(max_rounds=AUTO_MAX_ROUNDS)
display(Markdown(format_automatic_demo_result(auto_demo_result)))


## Simplified Planning vs. Guided and Automatic Planner-to-`ReactAgent` Loops

`03_lab_notebook.ipynb` keeps the planning pattern simplest by showing planning and replanning with manually provided observations.

In this extension notebook:
- **Part 1** is a guided two-round example of `PlanningAgent` assigning tasks to `ReactAgent` and then revising the plan.
- **Part 2** is a bounded automatic demo that can keep repeating the same pattern for up to `5` rounds or stop earlier when `PlanningAgent` returns `FINISHED:`.

All three views teach the same caution: students should not overclaim message delivery when the evidence only supports activity plus later connectivity context.
